# Phase 4: Dense Semantic Retrieval Using BERT & InLegalBERT

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Dataset**: `data/processed/IL-TUR-PCR/` (Prior Case Retrieval)  
**Objective**: Implement transformer-based dense semantic retrieval using attention-mask mean pooling, L2 normalization, and cosine similarity ranking.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Path setup
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings.model_loader import ModelLoader, SUPPORTED_MODELS
from src.embeddings.pooling import masked_mean_pooling, l2_normalize
from src.embeddings.embedding_generator import EmbeddingGenerator
from src.retrieval.dense_retriever import DenseRetriever
from src.evaluation.retrieval_metrics import evaluate_query_ranking
print("Dense retrieval modules imported successfully.")

## 1. Model Investigation & Architecture Specifications
We compare the three candidate transformer encoders: `bert-base-uncased`, `nlpaueb/legal-bert-base-uncased`, and `law-ai/InLegalBERT`.

In [ ]:
model_specs = []
for name, info in SUPPORTED_MODELS.items():
    model_specs.append({
        "Alias": name,
        "HF Model ID": info["hf_id"],
        "Domain": info["domain"],
        "Hidden Dim": info["hidden_size"],
        "Max Length": info["max_length"],
        "License": info["license"],
    })
display(pd.DataFrame(model_specs))

## 2. Attention-Mask-Aware Mean Pooling & L2 Normalization
Because transformer encoders output hidden states for every token (including padding), we apply masked mean pooling to weight only active tokens.

In [ ]:
# Load InLegalBERT model and tokenizer
model, tokenizer, device = ModelLoader.load_model_and_tokenizer("inlegalbert")
print(f"Active device: {device} | Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Sample legal text
sample_texts = [
    "The appellant was convicted under Section 302 of the Indian Penal Code for murder.",
    "Dispute under the Arbitration and Conciliation Act 1996 regarding commercial contract."
]
inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(**inputs)
    pooled = masked_mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    normed = l2_normalize(pooled)

print("Tokenized input shape:", inputs["input_ids"].shape)
print("Last hidden state shape:", outputs.last_hidden_state.shape)
print("Pooled & L2-normalized embedding shape:", normed.shape)
print("Embedding norm verification:", torch.norm(normed, dim=-1).tolist())

## 3. Load Cached Dense Embeddings & Dense Cosine Retrieval
We load the generated full-document embeddings for development queries and candidates.

In [ ]:
emb_dir = PROJECT_ROOT / "models" / "embeddings" / "inlegalbert"
cand_embs = np.load(emb_dir / "dev_candidates" / "fulldoc_embeddings.npy")
query_embs = np.load(emb_dir / "dev_queries" / "fulldoc_embeddings.npy")

cand_docs = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "IL-TUR-PCR" / "dev_candidates" / "documents.parquet")
query_docs = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "IL-TUR-PCR" / "dev_queries" / "documents.parquet")

print(f"Candidate Embeddings: {cand_embs.shape}")
print(f"Query Embeddings: {query_embs.shape}")

# Evaluate first query with query self-exclusion
sample_qid = query_docs.iloc[0]["id"]
ground_truth = set(query_docs.iloc[0]["relevant_candidates"])
print(f"\nSample Query ID: {sample_qid}")
print(f"Ground Truth Precedents: {list(ground_truth)[:5]}...")

# Cosine similarity (inner product of L2-normalized vectors)
sims = np.dot(query_embs[0], cand_embs.T)

from src.retrieval.document_retriever import DocumentRetriever
doc_retriever = DocumentRetriever(candidate_doc_ids=cand_docs["id"].values)
ranked = doc_retriever.rank_candidates(sims, query_id=sample_qid, top_n=5)

print("\n--- Top 5 Retrieved Precedents (InLegalBERT FullDoc) ---")
for rank, (cand_id, score) in enumerate(ranked, start=1):
    is_hit = "[RELEVANT HIT]" if cand_id in ground_truth else ""
    print(f"{rank}. Doc ID: {cand_id} | Cosine Sim: {score:.4f} {is_hit}")

## 4. Comprehensive Benchmark Comparison
We compare the dense retrieval models against Phase 3 lexical baselines (BM25 and TF-IDF).

In [ ]:
model_comp_csv = PROJECT_ROOT / "results" / "tables" / "dense_model_comparison.csv"
if model_comp_csv.exists():
    display(pd.read_csv(model_comp_csv))
else:
    print("Benchmark table will appear after evaluator completes.")